In [1]:
import re
import warnings
import pyreadr
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
import statsmodels.tools.sm_exceptions as sm_exceptions
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [10]:
# loading phenotype data
phenoPath = "C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/01uthealth/00MetaData_UTHealth_RNAseq-07292025.csv"
phenoData = pd.read_csv(phenoPath)
phenoData.rename(columns={phenoData.columns[0]: "SampleID"}, inplace=True)
phenoData = phenoData[['SampleID', 'SAB', 'UTID', 'RIN_novogene', 'BA9_RNAseq_Batch']]
phenoData
# Load phenotypes
pheno = pd.read_csv('C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/01uthealth/01MetaData_UTHealth_Methylation-08052024.csv')
# Merge on 'SampleID'
phenoData = pd.merge(phenoData, pheno, on='SAB')
# Load data
uthh = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/00uthhealth_rnaagecalc-07272025.csv")
# Rename the first column to SampleID
uthh.rename(columns={uthh.columns[0]: "SampleID"}, inplace=True)
uthh.rename(
    columns={col: f"{col}_clock" for col in uthh.columns if col != "SampleID"},
    inplace=True
)
# Load data
dpclo = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/00deepclock-08032025.csv")
# Rename and keep only SampleID and KPANN_brain columns
dpclo = dpclo.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain_clock'})[['SampleID', 'KPANN_brain_clock']]
# Merge with both filtered datasets
merged_df_com = pd.merge(uthh, dpclo, on='SampleID')
# Load cell types and SVAs
cells = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/05celltype-08042025/01uthealth_cellprop_svas-08042025.csv")
cells = cells.rename(columns={'name': 'SampleID'})
# Merge with both filtered datasets
merged_df_com = pd.merge(merged_df_com, cells, on='SampleID')
# Load KPANN clock trained in VABB for UTHealth
adicloc = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/01deepclock_UTHealth_only-08052025.csv")
adicloc = adicloc.rename(columns={'name': 'SampleID'})
# Merge with both filtered datasets
merged_df_com = pd.merge(merged_df_com, adicloc, on='SampleID')
# Merge on SampleID
data_assoc = pd.merge(phenoData, merged_df_com, on="SampleID", how="inner")
data_assoc.head()

,SampleID,SAB,UTID,RIN_novogene,BA9_RNAseq_Batch,Sentrix_ID,Year,Ethnicity,Gender,Age,...,W_2,W_3,W_4,W_5,W_6,W_7,W_8,W_9,W_10,KPANNtrainedVABB_brain_clock
0,A67900,67900,UTHBC0001,6.9,B2018,202787550060_R01C01,2018,White,Male,17,...,-0.011269,-0.032170,0.000214,-0.033074,0.001498,-0.049769,0.002038,0.004475,-0.029818,42.436069
1,A67902,67902,UTHBC0003,7.2,B2018,202787550060_R03C01,2018,Hispanic,Male,31,...,-0.002684,-0.009380,0.026539,-0.102646,0.045524,-0.048069,-0.167566,-0.106294,0.069254,48.787434
2,A67905,67905,UTHBC0006,8.4,B2018,202787550060_R04C01,2018,White,Male,43,...,-0.003179,-0.001653,-0.045735,-0.043083,-0.008536,0.033994,-0.014402,0.002389,0.036657,40.489330
3,A67906,67906,UTHBC0007,7.5,B2018,202787550060_R05C01,2018,White,Male,44,...,-0.007012,0.026808,0.028630,-0.036488,0.000311,-0.034601,0.024915,-0.043130,0.009967,44.178295
4,A67907,67907,UTHBC0008,7.6,B2018,202787550060_R06C01,2018,White,Male,53,...,0.003290,-0.034876,0.028161,0.024909,0.056005,0.014285,-0.107466,0.156924,0.006903,42.630310


In [11]:
# Standarizing all covariates
# Select only numeric columns
numeric_cols = data_assoc.select_dtypes(include='number').columns
# Scale
scaler = StandardScaler()
data_assoc[numeric_cols] = scaler.fit_transform(data_assoc[numeric_cols])

In [12]:
# Copy your data
binary_df = data_assoc.copy()

# Define clocks and phenotype columns
clock_cols = [col for col in binary_df.columns if col.endswith('_clock') and pd.api.types.is_numeric_dtype(binary_df[col])]
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD', 'Amphetamine', 'Benzodiazepine', 'Cannabis', 'Suicide', 'MDD', 'BD']
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['Age', 'Gender', 'RIN_novogene', 'PMIhrs', 'ast', 'end', 'mic', 'neu', 'oli', 'opc',' W_1', 'W_2']

# Columns to convert
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD',
                  'Amphetamine', 'Benzodiazepine', 'Cannabis',
                  'Suicide', 'MDD', 'BD']
cols_to_convert = phenotype_cols + ['Gender']

for col in cols_to_convert:
    if col in binary_df.columns:
        # If bool, convert directly to int
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df[col].dtype == 'object':
            if col == 'Gender':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df.columns:
        print(f"{col} unique values: {binary_df[col].unique()}")

## Running models
# Adding a dataframe to store the results
results = []
# Loooping in all phenotypes
for pheno in phenotype_cols:
    for clock in clock_cols:
        try:
            formula = f"{pheno} ~ {clock} + {' + '.join(covariates)}"
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")  # suppress all warnings
                
                # Also suppress specific statsmodels warnings if you want
                warnings.filterwarnings("ignore", category=sm_exceptions.ConvergenceWarning)
                warnings.filterwarnings("ignore", category=RuntimeWarning)
                
                model = smf.logit(formula=formula, data=binary_df).fit(disp=0)
            
            # Check convergence
            converged = model.mle_retvals.get('converged', True)
            
            if not converged:
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': np.nan,
                    'SE': np.nan,
                    'z-value': np.nan,
                    'P-value': np.nan,
                    'CI_lower_95': np.nan,
                    'CI_upper_95': np.nan,
                    'Converged': False
                })
                print(f"Model did NOT converge for {pheno} ~ {clock}")
            else:
                coef = model.params[clock]
                pval = model.pvalues[clock]
                conf_int = model.conf_int().loc[clock]
                zval = model.tvalues[clock]
                
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': coef,
                    'SE': model.bse[clock],
                    'z-value': zval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'Converged': True
                })
        except Exception as e:
            print(f"Model failed for {pheno} ~ {clock}: {e}")
            results.append({
                'Phenotype': pheno,
                'Clock': clock,
                'Beta': np.nan,
                'SE': np.nan,
                'z-value': np.nan,
                'P-value': np.nan,
                'CI_lower_95': np.nan,
                'CI_upper_95': np.nan,
                'Converged': False
            })

results_df = pd.DataFrame(results)

Overdose unique values: [0 1]
Psych_Diagnosis unique values: [0 1]
AUD unique values: [0 1]
OUD unique values: [0 1]
CUD unique values: [0 1]
Amphetamine unique values: [0 1]
Benzodiazepine unique values: [0 1]
Cannabis unique values: [0 1]
Suicide unique values: [0 1]
MDD unique values: [0 1]
BD unique values: [0 1]
Gender unique values: [1 0]
Model failed for Amphetamine ~ deMagalhaes_nerve_clock: Singular matrix
Model failed for Suicide ~ DESeq2_adipose_tissue_clock: Singular matrix
Model failed for Suicide ~ deMagalhaes_adipose_tissue_clock: Singular matrix
Model failed for Suicide ~ GenAge_adipose_tissue_clock: Singular matrix
Model failed for Suicide ~ Peters_adipose_tissue_clock: Singular matrix
Model failed for Suicide ~ all_adipose_tissue_clock: Singular matrix
Model failed for Suicide ~ DESeq2_adrenal_gland_clock: Singular matrix
Model failed for Suicide ~ Pearson_adrenal_gland_clock: Singular matrix
Model failed for Suicide ~ Dev_adrenal_gland_clock: Singular matrix
Model fa

In [13]:
### Storing results
# Extract tissue name from the clock column
def extract_tissue(clock_name):
    # Match the pattern: anything between two underscores and before "_clock"
    match = re.search(r'_(.*?)_clock$', clock_name)
    if match:
        return match.group(1)
    else:
        return None  # fallback in case the format doesn't match

# Add 'Tissue' column to results DataFrame
results_df['Tissue'] = results_df['Clock'].apply(extract_tissue)

# Save results with tissue information
results_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/02uthealth_assoc-08052025.csv", index=False)
results_df

,Phenotype,Clock,Beta,SE,z-value,P-value,CI_lower_95,CI_upper_95,Converged,Tissue
0,Overdose,DESeq2_adipose_tissue_clock,0.055710,0.315774,0.176424,0.859961,-0.563196,0.674617,True,adipose_tissue
1,Overdose,Pearson_adipose_tissue_clock,-0.553536,0.359364,-1.540320,0.123482,-1.257878,0.150805,True,adipose_tissue
2,Overdose,Dev_adipose_tissue_clock,-0.116157,0.473774,-0.245173,0.806323,-1.044737,0.812424,True,adipose_tissue
3,Overdose,deMagalhaes_adipose_tissue_clock,-0.247172,0.478147,-0.516937,0.605200,-1.184322,0.689978,True,adipose_tissue
4,Overdose,GenAge_adipose_tissue_clock,0.045852,0.528293,0.086792,0.930837,-0.989584,1.081287,True,adipose_tissue
...,...,...,...,...,...,...,...,...,...,...
2305,BD,GTExAge_vagina_clock,NaN,NaN,NaN,NaN,NaN,NaN,False,vagina
2306,BD,Peters_vagina_clock,NaN,NaN,NaN,NaN,NaN,NaN,False,vagina
2307,BD,all_vagina_clock,NaN,NaN,NaN,NaN,NaN,NaN,False,vagina
2308,BD,KPANN_brain_clock,NaN,NaN,NaN,NaN,NaN,NaN,False,brain


In [29]:
# Load Epigenetic clocks
# Load the .rds file
epiclocks = pd.read_csv('C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/03epigenclocks-08022025/01pheno_uthealth_epiclocks-08042025.txt', sep='\t')  # Replace with your file path
# Load cortical clock
rds_path = ('C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/03epigenclocks-08022025/00uthealth_brain-08032025.rds')  # Replace with your file path
result = pyreadr.read_r(rds_path)
# Extract the DataFrame (it's usually the first item in the returned dictionary)
df2 = list(result.values())[0]
df2 = df2[['SAB', 'corticalClock', 'Zhang2019']].rename(columns={'corticalClock': 'corticalClock_clock'})
# Merge on 'SAB'
merged_df2 = pd.merge(df, df2, on='SAB')
# Rename specific clock columns
merged_df2.rename(columns={
    'StochClock1': 'StocH',
    'StochClock2': 'StocP',
    'StochClock3': 'StocZ',
    'EAA_StochClock1': 'EAAStocH',
    'EAA_StochClock2': 'EAAStocP',
    'EAA_StochClock3': 'EAAStocZ',
}, inplace=True)

# Drop repeated columns
columns_to_drop = [
    "Sentrix_ID", "Year", "Ethnicity", "Gender", "Age", "PMIhrs", "pH",
    "Cause_of_Death", "Drugs_at_TOD", "Overdose.", "Diagnosis", "Opioid",
    "Cocaine", "Amphetamine", "Benzodiazepine", "Cannabis", "Alcohol",
    "Suicide", "Psych_Autopsy", "Consensus", "Consensus_Diagnoses_All",
    "Consensus_Dx_1", "Consensus_Dx_2", "Consensus_Dx_3", "Consensus_Dx_4",
    "Consensus_Dx_5", "Consensus_Dx_6", "Notes", "Sample_Name", "name"
]

# Assuming your DataFrame is called `df`
merged_df2 = merged_df2.drop(columns=columns_to_drop, errors='ignore')  # 'errors=ignore' skips any missing cols
merged_df2

,SAB,NeuN_neg,NeuN_pos,PC1,PC2,PC3,PC4,PC5,PC6,PC7,...,StocP,StocZ,EAAStocH,EAAStocP,EAAStocZ,CausAge,DamAge,AdaptAge,corticalClock_clock,Zhang2019
0,67917,0.573929,0.449426,-23999.961595,-6242.864130,-6665.884330,1293.030437,-5005.231205,2870.010837,-3640.157262,...,4.513590,38.723731,-3.166403,-0.996325,-5.125392,61.659157,89.769912,-17.427697,40.408642,30.096239
1,67927,0.624761,0.431789,-16929.244300,-7859.922078,-4262.488007,-3302.846974,-1485.839042,-6463.843525,-1715.236147,...,7.576214,45.290771,-2.723828,9.413272,2.261846,56.128294,98.530467,-25.842521,38.114053,25.519290
2,67928,0.540216,0.433918,-18329.421456,6634.048847,-988.625715,-8945.066684,-5556.492060,1126.866764,-4362.820621,...,1.795859,39.731010,1.094699,-9.224286,-4.733263,64.668921,90.596466,-3.609520,44.070327,33.184961
3,67929,0.554959,0.420519,-3165.670165,29211.332060,-2545.991761,-13250.496413,-2082.821465,-4780.325990,-5395.537133,...,2.873425,45.256590,-5.712535,2.261493,1.954265,57.276368,84.639845,1.553281,34.787823,30.346163
4,67930,0.578921,0.418987,-9780.332414,21925.809818,-2974.784448,-10869.579501,-5945.755229,-1059.951731,-6963.158441,...,3.694324,42.525289,0.355412,-4.264582,-1.597234,71.308171,88.779538,9.601219,46.062219,40.702236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,67984,0.523799,0.452060,41038.039275,-6779.180940,-3018.163130,-278.599678,-3248.327818,1585.056726,3066.833217,...,1.306424,45.815595,0.593450,0.082244,2.444921,64.455636,97.714300,-12.570473,37.612312,31.980921
110,79212,0.511119,0.438691,33644.785919,-18863.460345,-2515.783668,-3687.574238,2575.243834,-6968.877634,2944.167365,...,19.871303,46.616532,5.594370,1.504184,1.332061,67.530403,95.870614,-9.721639,68.109024,45.081531
111,79215,0.546922,0.423037,25828.549937,-16344.231363,4775.181800,-3356.952452,-1788.444292,2012.396380,2340.350387,...,1.355331,42.775063,5.702157,-4.154584,-1.074061,66.088791,93.359736,-17.125473,39.585491,33.981887
112,79238,0.543958,0.405637,42163.122666,-10626.262629,-11753.761041,1275.820812,2439.249005,-6047.091242,1735.723511,...,15.498145,42.140199,1.756163,-3.481222,-3.212622,73.700413,100.408326,-15.757594,62.622326,45.822374


In [45]:
# Load phenotypes
pheno = pd.read_csv('C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/00databases/00phenotype/01uthealth/01MetaData_UTHealth_Methylation-08052024.csv')
data_assoc2 = pd.merge(pheno, merged_df2, on='SAB')
data_assoc2 = data_assoc2.dropna()

# Standarizing all covariates
# Select only numeric columns
numeric_cols = data_assoc2.select_dtypes(include='number').columns
# Scale
scaler = StandardScaler()
data_assoc2[numeric_cols] = scaler.fit_transform(data_assoc2[numeric_cols])
data_assoc2

,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,COD,Overdose,Psych_Diagnosis,...,StocP,StocZ,EAAStocH,EAAStocP,EAAStocZ,CausAge,DamAge,AdaptAge,corticalClock_clock,Zhang2019
0,-0.667482,201465920012_R01C01,-0.930949,Hispanic,Male,-0.114734,0.790827,Intoxication,Yes,Yes,...,-0.188780,-1.562227,-0.644688,-0.167074,-1.611393,-0.135859,-0.397083,-0.492011,-0.228244,-0.321678
1,-0.665552,201465920012_R02C01,-0.930949,Black,Female,-0.930623,0.665061,Undetermined,No,Yes,...,0.088099,0.375073,-0.552878,1.481481,0.686503,-1.120618,1.061840,-1.224113,-0.383383,-0.783504
2,-0.665359,201465920012_R03C01,-0.930949,Black,Male,0.497182,0.331210,Intoxication,Yes,Yes,...,-0.434480,-1.265076,0.239257,-1.470127,-1.489416,0.400024,-0.259435,0.710190,0.019324,-0.010018
3,-0.665166,201465920012_R04C01,-0.930949,White,Female,-0.658660,0.279761,Intoxication,Yes,Yes,...,-0.337061,0.364989,-1.172870,0.348862,0.590826,-0.916206,-1.251410,1.159361,-0.608271,-0.296460
4,-0.664973,201465920012_R05C01,-0.930949,White,Male,0.157229,0.634191,Injuries,No,No,...,-0.262847,-0.440754,0.085896,-0.684665,-0.513914,1.582130,-0.562014,1.859544,0.153997,0.748494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,1.517110,205062740040_R04C01,1.074172,White,Female,-0.726651,-0.540003,Natural,No,Yes,...,-0.247998,-0.306108,-0.092022,0.601839,-0.095544,-0.090861,0.997428,-1.099672,-0.571790,-0.229929
108,1.517304,205062740040_R05C01,1.074172,White,Male,0.021247,-1.043066,Undetermined,No,Yes,...,-0.472752,-0.036211,0.045055,-0.858444,-0.044827,0.548792,-0.167768,0.064563,0.012882,-0.122728
109,1.517497,205062740040_R06C01,1.074172,White,Male,-0.522679,0.076249,Intoxication,Yes,Yes,...,-0.100943,0.828361,0.918837,0.568560,1.036900,0.204574,1.549996,-1.117848,-0.210165,-0.099818
110,1.517690,205062740040_R07C01,1.074172,White,Male,-0.794642,0.386090,Intoxication,Yes,Yes,...,-0.946814,-0.434266,0.675321,-0.525349,-0.209417,0.039717,0.799395,-0.783350,-0.663805,-0.551797


In [47]:
# Copy your data
binary_df = data_assoc2.copy()

# Define clocks and phenotype columns
clock_cols = ['Zhang2019', 'Horvath1', 'PhenoAge', 'StocH', 'StocP', 'StocZ', 'EAAStocH', 'EAAStocP', 'EAAStocZ', 'corticalClock_clock', 'CausAge', 'AdaptAge', 'DamAge']
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD', 'Amphetamine', 'Benzodiazepine', 'Cannabis', 'Suicide', 'MDD', 'BD']
# Convert Gender (and any other categorical covariates) to numeric BEFORE correlation
covariates = ['Age', 'Gender', 'NeuN_neg', 'NeuN_pos', 'PC1', 'PC2']

# Columns to convert
phenotype_cols = ['Overdose', 'Psych_Diagnosis', 'AUD', 'OUD', 'CUD',
                  'Amphetamine', 'Benzodiazepine', 'Cannabis',
                  'Suicide', 'MDD', 'BD']
cols_to_convert = phenotype_cols + ['Gender']

for col in cols_to_convert:
    if col in binary_df.columns:
        # If bool, convert directly to int
        if binary_df[col].dtype == 'bool':
            binary_df[col] = binary_df[col].astype(int)
        # If object/string, convert 'Yes'/'No' and also 'Male'/'Female' for Gender
        elif binary_df[col].dtype == 'object':
            if col == 'Gender':
                binary_df[col] = binary_df[col].map({'Male': 1, 'Female': 0})
            else:
                binary_df[col] = binary_df[col].map({'Yes': 1, 'No': 0})
        else:
            # If already numeric, no change
            pass

# Quick check of unique values after conversion
for col in cols_to_convert:
    if col in binary_df.columns:
        print(f"{col} unique values: {binary_df[col].unique()}")

## Running models
# Adding a dataframe to store the results
results = []
# Loooping in all phenotypes
for pheno in phenotype_cols:
    for clock in clock_cols:
        try:
            formula = f"{pheno} ~ {clock} + {' + '.join(covariates)}"
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")  # suppress all warnings
                
                # Also suppress specific statsmodels warnings if you want
                warnings.filterwarnings("ignore", category=sm_exceptions.ConvergenceWarning)
                warnings.filterwarnings("ignore", category=RuntimeWarning)
                
                model = smf.logit(formula=formula, data=binary_df).fit(disp=0)
            
            # Check convergence
            converged = model.mle_retvals.get('converged', True)
            
            if not converged:
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': np.nan,
                    'SE': np.nan,
                    'z-value': np.nan,
                    'P-value': np.nan,
                    'CI_lower_95': np.nan,
                    'CI_upper_95': np.nan,
                    'Converged': False
                })
                print(f"Model did NOT converge for {pheno} ~ {clock}")
            else:
                coef = model.params[clock]
                pval = model.pvalues[clock]
                conf_int = model.conf_int().loc[clock]
                zval = model.tvalues[clock]
                
                results.append({
                    'Phenotype': pheno,
                    'Clock': clock,
                    'Beta': coef,
                    'SE': model.bse[clock],
                    'z-value': zval,
                    'P-value': pval,
                    'CI_lower_95': conf_int[0],
                    'CI_upper_95': conf_int[1],
                    'Converged': True
                })
        except Exception as e:
            print(f"Model failed for {pheno} ~ {clock}: {e}")
            results.append({
                'Phenotype': pheno,
                'Clock': clock,
                'Beta': np.nan,
                'SE': np.nan,
                'z-value': np.nan,
                'P-value': np.nan,
                'CI_lower_95': np.nan,
                'CI_upper_95': np.nan,
                'Converged': False
            })

results_df = pd.DataFrame(results)

Overdose unique values: [1 0]
Psych_Diagnosis unique values: [1 0]
AUD unique values: [1 0]
OUD unique values: [0 1]
CUD unique values: [0 1]
Amphetamine unique values: [0 1]
Benzodiazepine unique values: [1 0]
Cannabis unique values: [0 1]
Suicide unique values: [0 1]
MDD unique values: [1 0]
BD unique values: [0 1]
Gender unique values: [1 0]


In [48]:
# Save results with tissue information
results_df.to_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/02uthealth_assoc_epiclocks-08052025.csv", index=False)